## CLIP Embedding

Run scraper.ipynb and then img_extractor for the necessary corpus. Use the image-extractor kernel.

This file takes the raw article TXTs and image JPGs and jointly embeds them for semantic and multimodal search. Text is cleaned and embedded using a sliding window method to improve retrieval. Vectors are stored in a FAISS index, with a parallel metadata file (JSON) mapping each index entry to its source issue, boundaries (for images), and mode (text/image).

In [1]:
# imports
import json
import csv
import re
import numpy as np
import faiss
from pathlib import Path
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel
from tqdm import tqdm

In [ ]:
# Input paths
BASE_DIR = Path("output")
TEXT_DIR = BASE_DIR / "plain_text"
IMG_DIR  = BASE_DIR / "images"
IMG_METADATA_CSV = BASE_DIR / "img_metadata.csv"

# New clip folder
CLIP_DIR = BASE_DIR / "clip"
CLIP_DIR.mkdir(parents=True, exist_ok=True)

# Storage paths
FAISS_INDEX_PATH = CLIP_DIR / "clip.index"
METADATA_PATH = CLIP_DIR / "clip_metadata.json"

In [ ]:
# Load CLIP
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model.eval()

In [ ]:
# Text cleaning
def clean_text(text):
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[|_]{2,}', '', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    return text

# Sliding window chunking
def chunk_text(text, max_tokens=70, overlap=10):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunks.append(' '.join(words[i:i + max_tokens]))
        i += max_tokens - overlap
    return chunks

# Embed batch of text strings
def embed_texts(texts):
    with torch.no_grad():
        inputs = processor(
            text=texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=77
        ).to(device)
        features = model.get_text_features(**inputs)
        features = features / features.norm(dim=-1, keepdim=True)
    return features.cpu().numpy()

# Embed single image
def embed_image(img_path):
    image = Image.open(img_path).convert("RGB")
    with torch.no_grad():
        inputs = processor(images=image, return_tensors="pt").to(device)
        features = model.get_image_features(**inputs)
        features = features / features.norm(dim=-1, keepdim=True)
    return features.cpu().numpy()

In [ ]:
# Set up FAISS index
dim = 512  # CLIP ViT-B/32
index = faiss.IndexFlatIP(dim)
metadata = []

### Embed and Store

In [ ]:
# TEXT
txt_paths = sorted(TEXT_DIR.rglob("*.txt"))
for txt_path in tqdm(txt_paths, desc="Embedding text"):
    try:
        text = clean_text(txt_path.read_text(encoding="utf-8"))
        chunks = chunk_text(text)
        if not chunks:
            continue

        year  = txt_path.parent.parent.name
        month = txt_path.parent.name
        doc_id = txt_path.stem

        embeddings = embed_texts(chunks)
        for j, (chunk, emb) in enumerate(zip(chunks, embeddings)):
            index.add(emb.reshape(1, -1))
            metadata.append({
                "mode": "text",
                "doc_id": doc_id,
                "year": year,
                "month": month,
                "path": str(txt_path),
                "chunk_index": j,
                "chunk_text": chunk
            }) # pages not stored, would need to add separate pipeline for txts

    except Exception as e:
        print(f"Failed text: {txt_path}\n  {e}")

# IMAGES
with open(IMG_METADATA_CSV, newline="") as f:
    img_rows = list(csv.DictReader(f))

for row in tqdm(img_rows, desc="Embedding images"):
    try:
        img_path = Path(row["out_path"])
        if not img_path.exists():
            continue

        emb = embed_image(img_path)
        index.add(emb.reshape(1, -1))
        metadata.append({
            "mode": "image",
            "doc_id": row["base_name"],
            "year": Path(row["out_path"]).parent.parent.name,
            "month": Path(row["out_path"]).parent.name,
            "path": str(img_path),
            "source_jpg": row["jpg_path"],
            "label": row["label"],
            "score": row["score"],
            "x1": row["x1"],
            "y1": row["y1"],
            "x2": row["x2"],
            "y2": row["y2"],
            "crop_h": row["crop_h"],
            "crop_w": row["crop_w"]
        }) # pages not stored but can be added from path name

    except Exception as e:
        print(f"Failed image: {row['out_path']}\n  {e}")

# SAVE
faiss.write_index(index, str(FAISS_INDEX_PATH))
METADATA_PATH.write_text(json.dumps(metadata, indent=2))

n_text  = sum(1 for m in metadata if m["mode"] == "text")
n_image = sum(1 for m in metadata if m["mode"] == "image")
print(f"Done. {index.ntotal} vectors indexed | {n_text} text chunks | {n_image} images")